# BPL Writing Dfns (Direct Functions)

In [1]:
!pip install -U basedpl

In [2]:
%load_ext basedpl.notebooks

In [3]:
import basedpl
basedpl.__version__

'0.1.91'

## Writing your own functions with braces

A *dfn* (say "dee-fun") is a function you write yourself, inside braces `{ }`. Trains are great for short functions, and you can assign a tacit function to a name. That name refers to the whole function, however; the train does not create named, argument-dependent intermediate values while it runs. A dfn can do that: in `{double←ω×2 ◇ double+1}`, `double` temporarily holds the intermediate value `ω×2` for the current call, then disappears when the call finishes. A dfn is therefore often clearer when a function needs several steps, local intermediate values, or a choice between answers. In this dialog we build up dfns one feature at a time.

**Where the name comes from.** "Dfn" is short for *direct function*. The name comes from Dyalog APL, where John Scholes introduced them in the 1990s. They are a lighter alternative to APL's older *tradfns* ("traditional functions"), which needed a header line and control words like `:If` and `:EndIf`. A dfn is just its body, written straight into braces. BPL keeps dfns and has no tradfns.

**Dfns and tacit functions.** These are the two ways to write your own function in BPL as we mentioned above. They differ in whether you name the data:

- A **dfn** is *explicit*: it mentions the data with `ω` and `α`, as in `{(+/ω)÷≢ω}`.
- A **tacit** function (a train) never mentions it, as in `+/÷≢`.

Both give an ordinary function you can store in a name. Prefer tacit when it's short and clear. Switch to a dfn when a function needs several steps, its own names, or conditional logic that chooses between different results.

BPL docs: [Dfns](https://answerdotai.github.io/basedpl/dfns.html)

Glyphs and ideas this dialog teaches:

| Glyph | Plain English |
|---|---|
| `{ }` | a dfn: a function written in braces |
| `ω` | inside braces: the right argument |
| `α` | inside braces: the left argument |
| `◇` or a new line | separates statements; the **last** statement is the answer |
| `test?answer` | a predicate: if `test` is `$t`, give `answer` |
| `;` | separates the choices in a dfn: try the next one if the test fails |
| `α←` | a default left argument, used when none is given |
| `∇` | inside braces: this same function (for recursion) |
| `"DOMAIN"::` | an error guard: give a fallback answer instead of an error |
| `⍶` | inside braces: the function to the left (makes an operator) |
| `⎕←` | print a value, then pass it on unchanged (for debugging) |
| `[α ω]` | a list of both arguments; with one argument, `α` drops out |
| `∞::` | an error guard that catches any kind of error |
| `$e.message` | inside a guard: the error's message |
| `]help` | show BPL's documentation for a glyph or name |

Other glyphs used along the way:

| Glyph | Plain English |
|---|---|
| `←` | store a value or function in a name |
| `⍝` | comment: BPL ignores the rest of the line |
| `+ - × ÷` | add, subtract, multiply, divide (`-ω` alone flips the sign) |
| `< > =` | compare: answers `$t` (true) or `$f` (false) |
| `+/` `×/` | total, product of a list |
| `⌊/` `⌈/` | smallest, biggest of a list |
| `≢` | count the items |
| `¨` | each: do a function to every item |
| `⍤` | rank: which pieces a function is given (`⍤0` items, `⍤1` rows) |
| `[… ◇ …]` | a table written row by row; `◇` starts the next row |
| `+/÷≢` | a train: total ÷ count, which is the mean |
| `↦` | `↦5` gives 1 2 3 4 5 |
| `["key":value]` | a record: values with names; `.key` reads one |
| `"…"` | text |
| `ₓ` | (in output) marks an exact number |

### The data

A week of midday temperatures (°C), Monday to Sunday.

In [4]:
%%bpl
temps←12 15 11 18 20 14 9     ⍝ Mon to Sun
temps

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

12 15 11 18 20 14 9

### Arguments

#### 1. The right argument: `ω`

- Inside braces, `ω` stands for whatever is on the right of the function.
- `{ω×2}3` runs the dfn with `ω` as 3, so it gives 6.
- Store a dfn in a name to use it again, like any other function.

In [5]:
%%bpl
{ω×2}3            ⍝ ω is 3: gives 6
double←{ω×2}      ⍝ store it
temps
double temps      ⍝ ω is the whole list

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

6

12 15 11 18 20 14 9

24 30 22 36 40 28 18

#### 2. The left argument: `α`

- `α` stands for whatever is on the left.
- `2{α+ω}3` gives 5: `α` is 2, `ω` is 3.

In [6]:
%%bpl
2{α+ω}3           ⍝ 2 + 3
plus←{α+ω}
10 plus 1 2 3     ⍝ 11 12 13

⍝ argument order can matter
6{α÷ω}3
3{α÷ω}6

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

5

11 12 13

2

0.5

### Inside the braces

#### 3. Several steps

- A dfn can hold several statements, one per line (or separated by `◇`).
- They run top to bottom, and the **last** statement gives the answer.
- `spread` finds the coldest, then the hottest, then the difference.

In [7]:
%%bpl
spread←{
 lo←⌊/ω       ⍝ coldest
 hi←⌈/ω       ⍝ hottest
 hi-lo        ⍝ the last line is the answer
}
spread temps  ⍝ 20 − 9

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

11

#### 4. Names inside stay inside

- A name stored inside a dfn (like `lo`) is *local*: it belongs to that call only.
- Here we store 100 in `lo` outside, then call `spread`. Inside, `spread` makes its own `lo`. The outside `lo` is still 100 afterwards.

In [8]:
%%bpl
lo←100            ⍝ a name outside
spread temps      ⍝ uses its own lo inside
lo                ⍝ still 100

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

11

100

#### 5. Give back several answers in a record

- The last statement can be anything, including a record (`["key":value]`, values with names).
- `summary` returns the coldest, hottest and mean together. `.hi` then reads one of them.
- The brackets round `(+/ω)` make the total happen first, before `÷`.

In [9]:
%%bpl
summary←{["lo":⌊/ω "hi":⌈/ω "mean":(+/ω)÷≢ω]}
summary temps
(summary temps).hi     ⍝ just the hottest

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

["lo":9 "hi":20 "mean":14.14285714]

20

### Choosing

#### 6. Choose an answer: `test?answer;other`

- `?` after a test means "if this is true, the answer is what follows".
- `;` separates the choices. If the test is false, BPL moves on to the next choice.
- `abs←{ω<0?-ω;ω}` reads: "if `ω` is below 0, the answer is `-ω` (sign flipped); otherwise it's `ω`".

In [10]:
%%bpl
abs←{ω<0?-ω;ω}    ⍝ make a number positive
abs ¯3            ⍝ 3
abs 5             ⍝ 5

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

3

5

This is often called a **ternary operator** in other languages: it chooses one of two values based on a condition. JavaScript uses the familiar `? :` form:

```javascript
const x = -3;
const answer = x < 0 ? -x : x;
```

This closely corresponds to BPL's `x<0?-x;x`: both put the test first and the true answer before the false answer, using `;` in BPL where JavaScript uses `:`.

#### 7. More than two choices, and `¨`

- Chain choices with more `;`. The tests are tried in order, and the last choice has no test: it's the "otherwise".
- A test must be **one** yes/no. `temps>18` is seven of them, so `label temps` would be an error.
- Use `¨` (each) to run the dfn on one temperature at a time.

In [11]:
%%bpl
label←{ω>18?"hot";ω>12?"mild";"cold"}
label 20          ⍝ one temperature
label¨temps       ⍝ each temperature in turn

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

"hot"

┌→────────────────────────────────────────────────┐
│ "cold" "mild" "cold" "mild" "hot" "mild" "cold" │
└∊────────────────────────────────────────────────┘

### More tools

#### 8. A default left argument: `α←`

- `α←14` gives `α` a value **only when no left argument was passed**.
- So `above temps` counts the days above 14, and `18 above temps` counts the days above 18.

In [12]:
%%bpl
temps
above←{α←14 ◇ +/ω>α}   ⍝ count the days above α
above temps            ⍝ above 14: 3 days
18 above temps         ⍝ above 18: 1 day

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

12 15 11 18 20 14 9

3ₓ

1ₓ

#### 9. A dfn that calls itself: `∇`

- Inside braces, `∇` means "this same function". A function that calls itself is *recursive*.
- Factorial: 5! is 5×4×3×2×1. So `fact ω` is `ω × fact (ω-1)`, and `fact 0` is 1, which stops the chain.
- `×/↦5` (multiply 1 to 5 together) checks the answer.

In [13]:
%%bpl
fact←{ω=0?1;ω×∇ω-1}   ⍝ stop at 0; otherwise ω × fact(ω-1)
fact 5                ⍝ 120
×/↦5                  ⍝ the same, without recursion

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

120

120

#### How `fact` is parsed and run

```apl
fact←{ω=0?1;ω×∇ω-1}
```

Inside the braces are two possible answers:

1. `ω=0?1` — if `ω` is 0, return 1.
2. `ω×∇ω-1` — otherwise, evaluated right to left: subtract 1, call this function (`∇`) again, then multiply by `ω`.

For `fact 3`:

```text
fact 3 → 3 × fact 2
       → 3 × (2 × fact 1)
       → 3 × (2 × (1 × fact 0))
       → 3 × (2 × (1 × 1))
       → 6
```

`fact 0` takes the first branch and stops the recursion; the waiting multiplications then finish.

#### 10. Catch an error: `::`

- Dividing text gives a `DOMAIN` error (the wrong kind of value).
- `"DOMAIN"::"not a number"` at the start of a dfn says: if a `DOMAIN` error happens later in this dfn, give back `"not a number"` instead.

In [14]:
%%bpl
safe←{"DOMAIN"::"not a number" ◇ ÷ω}   ⍝ one over ω, or a message
safe 4            ⍝ 0.25
safe "a"          ⍝ the message instead of an error

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

0.25

"not a number"

#### 11. A dfn that takes a function: `⍶`

- `⍶` stands for the **function** on the left of the dfn. Braces that use `⍶` make an *operator*, like `¨` or `⍣`.
- `twice←{⍶ ⍶ ω}` applies its function two times. `double twice 3` is double of double of 3, which is 12.
- In `double twice 3`, `⍶` is `double`, so the body behaves like the two-function train `double double`: `ω` goes to the rightmost `double` first, then that result passes to the leftmost `double`.

In [15]:
%%bpl
twice←{⍶⍶ ω}     ⍝ do the function two times
double twice 3   ⍝ 3 → 6 → 12

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

12

### Debugging

When a function gives a surprising answer, the quickest fix is usually to **look at what it's doing**. Most of the tricks below come from Adám Brudzewsky in [APL Study Session 11](https://youtu.be/S9cfN9oVI5I), translated from Dyalog APL into BPL. Steps 12, 13, 19 and 20 are extra.

#### 12. Peek at a value: `⎕←`

- `⎕←x` prints `x`, then passes it on unchanged. Put it anywhere in a line to see a value on its way through.
- `+/⎕←temps>14` prints the yes/no answers, then adds them up.

In [16]:
%%bpl
+/⎕←temps>14      ⍝ prints $f $t …, then the count

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

$f $t $f $t $t $f $f

3ₓ

#### 13. Peek inside a dfn

- The same trick works on the lines of a dfn. `spreadP` is `spread` from step 3 with `⎕←` added, so it prints `lo` and `hi` as it goes. (P for print.)

In [17]:
%%bpl
spreadP←{
 ⎕←ω
 lo←⎕←⌊/ω     ⍝ print the coldest
 hi←⎕←⌈/ω     ⍝ print the hottest
 hi-lo
}
spreadP temps     ⍝ prints temps, low:9, then hi:20, then gives the result:11

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

12 15 11 18 20 14 9

9

20

11

#### 14. See what a function is given

- Wrap the function in braces that first print `[α ω]` (a list of the left and right arguments), then do the real work.
- Here the real work is `α=ω`. The printout shows that `=` gets all of `3 4` and all of `3 5` at once, then compares them item by item.

In [18]:
%%bpl
3 4{⎕←[α ω] ◇ α=ω}3 5      ⍝ prints both arguments, then $t $f

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────────┐
│ ┌→──┐ ┌→──┐ │
│ │3 4│ │3 5│ │
│ └~──┘ └~──┘ │
└∊────────────┘

$t $f

#### 15. See what rank hands it

- `⍤0` (rank 0) hands the function one item from each side at a time.
- So the wrapped function now prints **twice**: first 3 with 3, then 4 with 5.

In [19]:
%%bpl
3 4({⎕←[α ω] ◇ α=ω}⍤0)3 5  ⍝ one pair at a time

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

3 3

4 5

$t $f

#### 16. A trace operator: `tc`

- Rather than rewrite the braces each time, make an operator (step 11): `tc←{⎕←[α ω] ◇ α ⍶ ω}`. It prints the arguments, then runs its function `⍶` on them.
- Put `tc` straight after any function to trace it: `3 4=tc 3 5`.
- It works with one argument too. When there's no left argument, `α` simply drops out of `[α ω]` and of `α ⍶ ω`. (In Dyalog APL, Adám adds `α←⊢` to make that work. BPL doesn't need it.)

In [20]:
%%bpl
tc←{⎕←[α ω] ◇ α ⍶ ω}   ⍝ print the arguments, then run the function
3 4=tc 3 5             ⍝ two arguments
-tc 3 5                ⍝ one argument: only ω is printed

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────────┐
│ ┌→──┐ ┌→──┐ │
│ │3 4│ │3 5│ │
│ └~──┘ └~──┘ │
└∊────────────┘

$t $f

┌→──────┐
│ ┌→──┐ │
│ │3 5│ │
│ └~──┘ │
└∊──────┘

¯3 ¯5

#### How BPL reads `3 4=tc 3 5`

BPL already knows what each name denotes. This definition makes `tc` an **operator** because its dfn uses `⍶`, the function operand:

```apl
tc←{⎕←[α ω] ◇ α ⍶ ω}
```

For `3 4=tc 3 5`, the order is:

1. **Derive a function:** the operator `tc` takes the function immediately to its left, `=`. Thus `=tc` becomes a new function, with `⍶` standing for `=`.
2. **Call that function:** BPL groups the expression as `3 4 (=tc) 3 5`, so `α` is `3 4` and `ω` is `3 5`.
3. **Run the dfn:** statements run top to bottom:
   - `⎕←[α ω]` prints both arguments and passes the value on.
   - `α ⍶ ω` becomes `3 4 = 3 5`, producing `$t $f`.

So operators consume functions and produce **derived functions**; those derived functions then receive data normally. Within each expression, BPL evaluates functions right to left.

#### 17. Rank is just blinders

- `w` holds two weeks of temperatures, one week per row. (`[… ◇ …]` writes a table row by row.)
- `mean←+/÷≢` is total ÷ count. `mean tc w` shows that `mean` sees the **whole table**. `≢` then counts 2 rows, so the answer is wrong.
- `mean tc⍤1 w` shows `mean` getting **one row at a time**, so each week gets its own correct mean.
- As Adám puts it, rank is "just blinders": it limits how much of the data the function can see.

In [21]:
%%bpl
w←[12 15 11 18 20 14 9◇13 16 10 17 21 15 8]   ⍝ two weeks, one per row
mean←+/÷≢
mean tc w       ⍝ sees the whole table: wrong
mean tc⍤1 w     ⍝ sees one week at a time: right

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→──────────────────────┐
│ ┌→──────────────────┐ │
│ ↓12 15 11 18 20 14 9│ │
│ │13 16 10 17 21 15 8│ │
│ └~──────────────────┘ │
└∊──────────────────────┘

49.5 50

┌→──────────────────────┐
│ ┌→──────────────────┐ │
│ │12 15 11 18 20 14 9│ │
│ └~──────────────────┘ │
└∊──────────────────────┘

┌→──────────────────────┐
│ ┌→──────────────────┐ │
│ │13 16 10 17 21 15 8│ │
│ └~──────────────────┘ │
└∊──────────────────────┘

14.14285714 14.28571429

#### 18. Label the trace

- While not necessary, a record (`["key":value]`) makes the printout easier to read: `tl` prints `α` and `ω`.
- With one argument, the label `α` entry drops out along with the associated argument.

In [22]:
%%bpl
tl←{⎕←["α":α "ω":ω] ◇ α ⍶ ω}   ⍝ trace with labels
3 4=tl 3 5
-tl 3 5

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────────────┐
│ α:┌→──┐ ω:┌→──┐ │
│   │3 4│   │3 5│ │
│   └~──┘   └~──┘ │
└∊────────────────┘

$t $f

┌→────────┐
│ ω:┌→──┐ │
│   │3 5│ │
│   └~──┘ │
└∊────────┘

¯3 ¯5

#### 19. Show the error message instead of stopping

- `∞::` is an error guard (step 10) that catches **any** kind of error.
- Inside the guard, `$e.message` is the error's message as text.
- `why` is an **operator**, not an ordinary function, because its dfn uses `⍶`. In `label why temps`, `label` is the function supplied on the left, so `⍶` means `label`.
- `why` runs its function and, if that fails, gives back the message. Here it explains why `label temps` fails: a test needs **one** yes/no, but `temps>18` gives seven (step 7).

In [23]:
%%bpl
why←{∞::$e.message ◇ ⍶ ω}   ⍝ run the function; on error, give the message
label⁝why temps             ⍝ the message
label⁝why 20                ⍝ no error: the normal answer

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

"a predicate requires a Boolean singleton"

"hot"

#### 20. Look it up: `]help`

- `]help` followed by a glyph or name shows BPL's documentation for it, right in the notebook.

In [24]:
%%bpl
]help ⎕

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

# `⎕` — IO

Syntax. Keys: `Alt-q`. Hanzi: `出/入`

`⎕←Y` explicitly prints `Y` and retains its value.

```bpl
≥⎕←2              ⍝ 3
```

The example prints `2` before returning `3`. Explicit output is separate from implicit result display and is captured by the evaluation API.

## Input

Reading `⎕` gives the next line of standard input as a string, without its line ending. At the end of the input it gives `⍬`, not the `""` of an empty line. `⍎⎕` evaluates the line, as Dyalog's `⎕` does. `•nget "-"` gives the rest of standard input as one string.

```bash
printf '3\n4\n' | bpl -e '+/⍎¨[⎕ ⎕]'       # 7
```

`"-" •nput text` writes `text` to standard output with no line ending added. When `⎕` is read, the text written since the last line ending is its prompt. A terminal shows the prompt and reads the answer on the same line, and a notebook shows the prompt beside its text box.

```bash
printf 'bob\n' | bpl -e '"-" •nput "Name? " ◇ "Hi ",⎕'   # Name? Hi bob
```

`bpl -e`, script files and the interactive REPL read standard input. In a notebook, `⎕` asks for a line in a text box, but `•nget "-"` is an `IO` error. The Python API reads each line with Python's `input()`, and `•nget "-"` reads `sys.stdin`. `bpl -` and piped REPL input read the program itself from standard input. In those two modes and in `bpl --worker`, reading `⎕` or `•nget "-"` is an `IO` error.

Dyalog's `⎕A` and `⎕D` are BPL's [`$a` and `$d`](system/constants.qmd). `⎕C` and `⎕UCS` are [`•c`](system/c.qmd) and [`•ucs`](system/ucs.qmd).


### Your turn

1. Write `pct←{…}` so that `3 pct 12` gives 25 (3 is 25% of 12).
   *Hint:* `100×α÷ω`.
2. Write `move` that gives `"up"`, `"down"` or `"same"` for a change in temperature. Try `move¨1 ¯2 0`.
   *Hint:* two tests, `ω>0?` and `ω<0?`, then a last choice with no test.
3. Give `pct` a default left argument of 1, so `pct 4` gives 25.
   *Hint:* `α←1 ◇` at the start.
4. Write `total` with `∇` so that `total 5` gives 1+2+3+4+5, and check it with `+/↦5`.
   *Hint:* copy `fact`, but stop at 0 with 0, and use `+` instead of `×`.
5. Write an operator `thrice` and check that `double thrice 1` gives 8.
   *Hint:* copy `twice` and add one more `⍶`.
6. Use `tc` and `⍤1` to see what `⌈/` is given for each week of `w`.
   *Hint:* `⌈/tc⍤1 w`. It should print two rows, then give `20 21`.
7. Copy `above` (step 8) and add `⎕←α` so it prints the left argument it uses. Try it with and without a left argument.
   *Hint:* `{α←14 ◇ ⎕←α ◇ +/ω>α}`.